<a href="https://colab.research.google.com/github/ga4gh/analytics-dashboard/blob/plenary-proof-of-concept/notebooks/Analytics_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analytics Dashboard

This notebook provides a proof of concept for a comprehensive analytics dashboard for exploring GA4GH initiatives.

### What This Notebook Does

This notebook automatically:
1. **Fetches**
     * GitHub repositories
     * PyPi libraries
     * Pubmed articles
2. **Analyzes the data** to show trends, statistics, and insights
3. **Creates interactive visualizations** to graphically present trends in the data

## How to Run This Notebook

1. **Click "Run All" in your Jupyter environment** - Everything will run automatically
2. **Wait for results** - The notebook will fetch data and create visualizations
3. **Scroll through the results** - Each section provides different insights

## What You Can Customize

### Advanced Changes
- Requires knowledge of Plotly and Dash libraries
- Can modify visualization types, styling, and data presented

## What You'll See

1. **Yearly Productivity** - A chart of repositories, articles and libraries created every year since the inception of GA4GH
   
## Important Notes

- **Processing time** - Initial run may take 2-3 minutes
- **Internet required** - Fetches data from API set up by the tech team
- **No data storage** - Results are temporary unless you save them

## Troubleshooting

**If visualizations don't appear:**
1. Ensure you have the required libraries installed
2. Try refreshing your browser
3. Check that JavaScript is enabled

**If results seem incomplete:**  
***Note: This is not a complete dataset. This is a POC and will be more complete in the future***
1. Try running individual sections instead of all at once


### Libraries Used in This Notebook

**dash** → Used to build interactive dashboard components such as graphs, layouts, and user inputs directly inside the notebook.

**requests** → Connects to the Analytics Dashboard DB which is curated by the Tech Team and retrieves data in JSON format.

**pandas** → Helps convert API responses into DataFrames for easy filtering, analysis, and visualization.

**json** → Used for handling and inspecting raw JSON responses.

**typing (List, Optional, Dict, Any)** → Adds optional type hints to improve code readability and maintainability.

**plotly.express** → Creates interactive charts (bar, line, pie, scatter) quickly and easily.

In [3]:
# Make sure Dash is installed
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 60.9 MB/s eta 0:00:00


In [11]:
# Some of the imports required to use the notebook
import requests
import json
import dash
from typing import List, Optional, Dict, Any
from dash import Dash, dcc, html, Input, Output, callback, dash_table, jupyter_dash
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd


## Configurations & API Setup
---
**Explanation:**
- ```GH_BASE_URL```: is the main endpoint we will be using for this notebook. It will return a github repo by name as it is stored in the DB
- ```get_repos(GH_BASE_URL, repos_list)```: This function will retrieve all the repos under the "ga4gh" org from the DB
  1. Makes a GET request to the specified endpoint.  
  2. Raises an error if the request fails.  
  3. Returns the response as JSON for further processing.

---

**Data Exploration:**  
- You can test this by running, for example:  
  ```python
  get_repos(GH_BASE_URL, repos_list)

In [5]:
# Base URL
BASE_URL = "http://analytics-staging.ga4gh.org:8000"

# Github Endpoints
REPO_ENDPOINT = f"{BASE_URL}/github/name/"
GH_repos = "https://api.github.com/orgs/ga4gh/repos" # TEMP DELETE LATER
repos_list = ['ga4gh-schemas', 'ga4gh-server', 'compliance', 'dwg-website', 'gastore', 'mme-apis', 'metadata-team', 'g2p-team', 'benchmarking-tools', 'build', 'tool-registry-service-schemas', 'swg-ssig', 'tool-registry-server', 'workflow-execution-server', 'task-execution-server', 'task-execution-schemas', 'workflow-execution-service-schemas', 'ga4gh-consent-policy', 'hgvs-lib', 'tool-registry-validator', 'tool-registry-reference-implementation', 'vrs', 'hackathon2016', 'ga4gh-client', 'ga4gh-common', 'htsget', 'data-repository-service-schemas', 'ga4gh-converters', 'ADA-M', 'wiki', 'vrs-python', 'cloud-interop-testing', 'phenopacket-schema', 'large-scale-genomics-wiki', 'refget-compliance-suite', 'va-spec', 'variant-representation', 'approval-tracker', 'duri', 'refget-client', 'cloud-conformance-testing', 'data-object-service-schemas', 'w3id.org', 'data-security', 'pedigree', 'gh-openapi-docs', 'ga4gh-drs-client', 'refget-cloud', 'htsget-refserver', 'refget-loader', 'cloud-interop-ui', 'ga4gh-copyright-policy', 'TASC', 'standards-schema', 'htsget-compliance', 'ga4gh-registry', 'gatk', 'refget-compliance', 'htsjdk', 'ga4gh-bed', 'fasp-scripts', 'htsget-refserver-utils', 'ga4gh-ci', 'refget', 'reverse-lookup-spec', 'ga4gh-starter-kit-drs', 'vrs-vcf-alignment', 'pedigree-fhir-ig', 'vrsatile', 'ga4gh-starter-kit-docs', 'ga4gh-starter-kit-common', 'ga4gh-starter-kit-passport-broker', 'pedigree_family_history_terminology', 'ga4gh-starter-kit-wes', 'ga4gh-starter-kit', 'ga4gh-starter-kit-ui', 'machine-readable-consent-guidance', 'pedigree-tools', 'vrs-protobuf', 'ga4gh-starter-kit-utils', 'vrsatile-pydantic', 'ga4gh-testbed-lib', 'ga4gh-starter-kit-refget', 'ga4gh-testbed-ui', 'pedigree-validator', 'gks-metaschema', 'future-of-vcf', 'ga4gh-starter-kit-data-connect', 'cloud-best-practices', 'schema-registry-api', 'schema-registry-ui', 'schema-registry', 'tech-team', 'ga4gh-testbed-api', 'vrs-hackathons', 'Get-Started-with-GA4GH-APIs', 'ga4gh-starter-kit-passport-ui', 'vrs-phenopackets', 'cohort-rep-hackathon', 'product-process', 'openapi-test-runner', 'drs-compliance-suite', 'vrs-clojure', 'fasp-clients', 'sa-spec', 'gk-pilot', 'ga4gh-starter-kit-beacon', 'quality-control-wgs', 'Strategic-Refresh', 'drs-test-a-thon', 'experiments-metadata', 'ga4gh-pgx', 'compliance-tests-ga4gh-tes', 'ga4gh-crypt4gh', 'compliance-tests-ga4gh-wes', 'ga4gh-doi', 'gks-core', 'cat-vrs', 'compliance-tests-ga4gh-service-registry', 'ga4gh-testbed-api-aws-stack', 'gks2clinvar', 'human-pangenome-project', 'gks-validator', 'gks-portal', 'cat-vrs-python', 'va-spec-python', 'Website-flowcharts', 'gks-starter-repo']

# PyPi Endpoints
TOTAL_PACKAGES_ENDPOINT = f"{BASE_URL}/pypi/total-packages"
PACKAGE_VERSIONS_ENDPOINT = f"{BASE_URL}/pypi/package-versions"
RELEASES_OVER_YEARS_ENDPOINT = f"{BASE_URL}/pypi/releases-over-years"
ALL_SOURCES_COVERAGE_ENDPOINT = f"{BASE_URL}/pypi/all_sources_coverage"
PYPI_DETAILS_ENDPOINT = f"{BASE_URL}/pypi/project_details"

# PubMed Endpoints
PM_KEYWORD = f"{BASE_URL}/pubmed/articles/GA4GH"

# Function that will take a list of repos and retrieve them from the DB
def get_repos(endpoint: str, list_of_repos: List[str]) -> List[Dict[str, Any]]:

    items: List[Dict[str, Any]] = []
    url = endpoint

    for repo in list_of_repos:
        resp = requests.get(url + repo)
        resp.raise_for_status()
        items.append(resp.json()[0])

    return items

def get_json(endpoint: str, token: Optional[str] = None, per_page: int = 100) -> List[Dict[str, Any]]:

    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"

    params = {"per_page": per_page, "page": 1}
    items: List[Dict[str, Any]] = []
    url = endpoint

    while True:
        resp = requests.get(url, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if not isinstance(data, list):
            return data

        items.extend(data)

        if "next" in resp.links:
            url = resp.links["next"]["url"]
            params = None
        else:
            break

    return items

In [12]:
# -----------------------------------------------------
# 1. Total packages
# -----------------------------------------------------
total_packages_resp = get_json(TOTAL_PACKAGES_ENDPOINT)
total_packages = int(total_packages_resp.get("total_packages", 0))

# -----------------------------------------------------
# 2. Project details
# -----------------------------------------------------
details_data = get_json(PYPI_DETAILS_ENDPOINT)

# Build DataFrame
rows = []
for pkg in details_data:
    versions_list = pkg.get("versions", [])
    versions_count = versions_list[0].get("versions", 0) if versions_list else 0
    rows.append({
        "project_name": pkg.get("project_name"),
        "description": pkg.get("description"),
        "author_name": pkg.get("author_name"),
        "author_email": pkg.get("author_email"),
        "category": pkg.get("category"),
        "versions_count": versions_count,
    })

df = pd.DataFrame(rows)

# -----------------------------------------------------
# 3. Static Table (Replacing Dynamic DataTable)
# -----------------------------------------------------
# Since we cannot have an interactive Dash table without app.run,
# we create a Plotly Table to visualize the data structure.
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(df.columns),
        fill_color='#34495E',
        font=dict(color='white', size=12),
        align='left'
    ),
    cells=dict(
        values=[df[col] for col in df.columns],
        fill_color='#f9f9f9',
        align='left',
        font=dict(color='black', size=11)
    )
)])

fig_table.update_layout(
    title_text=f"Total PyPI Packages: {total_packages} (Data Preview)",
    margin=dict(l=20, r=20, t=50, b=20),
    height=400
)

# -----------------------------------------------------
# 4. Bar graph (Logic adapted from update_bar)
# -----------------------------------------------------
# Mimicking the "Page 1" logic from the original app (first 10 rows)
page_data = df.iloc[0:10]

# Create hover text: shorten long description
hover_texts = [
    f"<b>{row['project_name']}</b><br>"
    f"Category: {row.get('category', '')}<br>"
    f"Versions: {str(row.get('versions_count', ''))}"
    for _, row in page_data.iterrows()
]

fig_bar = go.Figure(data=[
    go.Bar(
        x=page_data["project_name"],      # X-axis: package names
        y=page_data["versions_count"],    # Y-axis: version counts
        marker={"color": "#2E86C1"},      # Bar color
        hovertext=hover_texts,            # Custom hover info
        hoverinfo="text"                  # Only show hover text
    )
])

fig_bar.update_layout(
    title={
        "text": "Package Versions Count (Top 10 Preview)",
        "x": 0.5,          # center title
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#2C3E50"}
    },
    xaxis={"title": "project_name", "tickangle": -45},
    yaxis={"title": "versions_count"},
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="#ffffff",
    margin={"b": 120}
)

# -----------------------------------------------------
# 5. Category distribution (Logic adapted from update_category_distribution)
# -----------------------------------------------------
# Count categories
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]

fig_pie = go.Figure(data=[
    go.Pie(
        labels=cat_counts["category"],
        values=cat_counts["count"],
        hole=0.4,
        textinfo="label+percent",
        hoverinfo="label+value+percent"
    )
])

fig_pie.update_layout(
    title={
        "text": "Category Distribution",
        "x": 0.5,          # center title
        "xanchor": "center",
        "yanchor": "top",
        "font": {"size": 20, "color": "#2C3E50"}
    },
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="#ffffff"
)

# Display figures
fig_table.show()
fig_bar.show()
fig_pie.show()

In [6]:
# 1. Process GitHub Data
GA4GH_repos = get_repos(REPO_ENDPOINT, repos_list)
gh_names = [repo['name'] for repo in GA4GH_repos]
gh_dates = [repo['pushed_at'] for repo in GA4GH_repos]

df_gh = pd.DataFrame({
    "item": gh_names,
    "created_at": pd.to_datetime(gh_dates, utc=True)
})
df_gh["Source"] = "GitHub Repos"

# 2. Process PubMed Data
pubmed_releases = get_json(PM_KEYWORD)
pm_names = [release['title'] for release in pubmed_releases]
pm_dates = [release['publish_date'] for release in pubmed_releases]

df_pm = pd.DataFrame({
    "item": pm_names,
    "created_at": pd.to_datetime(pm_dates, utc=True, errors='coerce')
})
df_pm["Source"] = "PubMed Articles"

# 3. Process PyPI Data (Pre-aggregated)
pypi_data = get_json(RELEASES_OVER_YEARS_ENDPOINT)
pypi_list = pypi_data.get("releases_over_years", [])

# Create DataFrame directly from the list of dicts
df_pypi = pd.DataFrame(pypi_list) # columns will be "year" and "releases"
df_pypi = df_pypi.rename(columns={"releases": "yearly_count"})
df_pypi["Source"] = "PyPI Releases"
# PyPI data doesn't have item names, so we set an empty string or message
df_pypi["items_str"] = "(Item names not available for PyPI)"
df_pypi["item"] = [[] for _ in range(len(df_pypi))] # Empty list for consistency

# 4. Combine GitHub and PubMed First
df_raw = pd.concat([df_gh, df_pm])
df_raw = df_raw.dropna(subset=["created_at"])
df_raw["year"] = df_raw["created_at"].dt.year

# Group GitHub/PubMed by Year and Source
grouped_raw = df_raw.groupby(["year", "Source"]).agg({"item": list}).reset_index()
grouped_raw["yearly_count"] = grouped_raw["item"].apply(len)
grouped_raw["items_str"] = grouped_raw["item"].apply(lambda items: "<br>".join(items))

# 5. Combine All DataFrames (GitHub/PubMed grouped + PyPI)
final_df = pd.concat([grouped_raw, df_pypi], ignore_index=True)

# Sort values
final_df = final_df.sort_values(["Source", "year"])

# 6. Create Graph
fig2 = px.line(
    final_df,
    x="year",
    y="yearly_count", # Use yearly count directly
    color="Source",
    markers=True,
    title="Updates per Year: GitHub, PubMed & PyPI",
    labels={"year": "Year", "yearly_count": "Updates per Year"},
    custom_data=["items_str", "Source"]
)

# 7. Update Hover Template
fig2.update_traces(
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>" # Source Name
        "Year: %{x}<br>"
        "Updates: %{y}<br><br>"
        "<b>Updated Items:</b><br>%{customdata[0]}<extra></extra>"
    ),
    marker=dict(size=8),
)

fig2.update_layout(
    xaxis_title="Year",
    yaxis_title="Updates per Year", # Axis Title
    margin=dict(l=40, r=20, t=70, b=120),
    height=500,
    legend_title_text=""
)